In [ ]:
import pandas as pd
# from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [ ]:
df = pd.read_csv('/content/healthinsurance.csv')

In [ ]:
df.head()

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title,claim
0,60.0,male,64,24.3,NoDisease,1,0,NewYork,72,0,0,Actor,13112.6
1,49.0,female,75,22.6,NoDisease,1,0,Boston,78,1,1,Engineer,9567.0
2,32.0,female,64,17.8,Epilepsy,2,1,Phildelphia,88,1,1,Academician,32734.2
3,61.0,female,53,36.4,NoDisease,1,1,Pittsburg,72,1,0,Chef,48517.6
4,19.0,female,50,20.6,NoDisease,0,0,Buffalo,82,1,0,HomeMakers,1731.7


In [ ]:
df.isna().sum()

,0
age,396
sex,0
weight,0
bmi,956
hereditary_diseases,0
no_of_dependents,0
smoker,0
city,0
bloodpressure,0
diabetes,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  14604 non-null  float64
 1   sex                  15000 non-null  object 
 2   weight               15000 non-null  int64  
 3   bmi                  14044 non-null  float64
 4   hereditary_diseases  15000 non-null  object 
 5   no_of_dependents     15000 non-null  int64  
 6   smoker               15000 non-null  int64  
 7   city                 15000 non-null  object 
 8   bloodpressure        15000 non-null  int64  
 9   diabetes             15000 non-null  int64  
 10  regular_ex           15000 non-null  int64  
 11  job_title            15000 non-null  object 
 12  claim                15000 non-null  float64
dtypes: float64(3), int64(6), object(4)
memory usage: 1.5+ MB


In [ ]:
from sklearn.impute import KNNImputer

num_cols = ['age', 'weight', 'bmi', 'no_of_dependents', 'smoker', 'bloodpressure', 'diabetes', 'regular_ex', 'claim']

imputer = KNNImputer(n_neighbors=5)
df[num_cols] = imputer.fit_transform(df[num_cols])

In [ ]:
df.isna().sum()

,0
age,0
sex,0
weight,0
bmi,0
hereditary_diseases,0
no_of_dependents,0
smoker,0
city,0
bloodpressure,0
diabetes,0


In [ ]:
df_feat = df.copy()

In [ ]:
# feature 1: age group
def age_group(age):
  if age < 25:
    return "young"
  elif age < 45:
    return "adult"
  elif age < 60:
    return "middle_aged"
  else: return "senior"


In [ ]:
df_feat['age_group'] = df_feat['age'].apply(age_group)


In [ ]:
# feature 2: lifestyle risk
def lifestyle_risk(row):
  if row['smoker']  and row['bmi'] > 30:
    return 'high'
  elif row['smoker'] or row['bmi'] > 27:
    return 'medium'
  else:
    return 'low'

In [ ]:
df_feat['lifestyle_risk'] = df_feat.apply(lifestyle_risk, axis=1)

In [ ]:
tier1 = [
    'NewYork', 'Boston', 'Philadelphia', 'WashingtonDC', 'Baltimore',
    'Atlanta', 'Charlotte', 'Miami', 'Tampa', 'Houston', 'Dallas',
    'Chicago', 'Minneapolis', 'SanFrancisco', 'SanJose', 'LosAngeles',
    'SanDiego', 'Phoenix', 'LasVegas', 'Denver', 'Portland', 'Orlando',
    'Nashville', 'NewOrleans', 'Cleveland', 'Columbus', 'Raleigh',
    'Louisville', 'KansasCity'
]

tier2 = [
    'Pittsburg', 'Buffalo', 'AtlanticCity', 'Cambridge', 'Hartford',
    'Springfield', 'Syracuse', 'York', 'Trenton', 'Warwick', 'Providence',
    'Harrisburg', 'Newport', 'Stamford', 'Worcester', 'Brimingham',
    'Charleston', 'Memphis', 'Macon', 'Huntsville', 'Knoxville',
    'Florence', 'PanamaCity', 'Kingsport', 'Marshall', 'Mandan',
    'Waterloo', 'IowaCity', 'Columbia', 'Indianapolis', 'Cincinnati',
    'Bloomington', 'Salina', 'Brookings', 'Minot', 'Lincoln', 'FallsCity',
    'GrandForks', 'Fargo', 'Canton', 'Rochester', 'JeffersonCity',
    'Escabana', 'Youngstown', 'SantaRosa', 'Eureka', 'Oxnard',
    'Oceanside', 'Carlsbad', 'Montrose', 'Prescott', 'Fresno', 'Reno',
    'Tucson', 'SanLuis', 'Kingman', 'Bakersfield', 'Mexicali', 'SilverCity',
    'SantaFe', 'Lovelock', 'Georgia', 'Oklahoma'
]


In [ ]:
# feature 3: city tier
def city_tier(city):
  if city in tier1:
    return 1
  elif city in tier2:
    return 2
  else:
    return 3

df_feat['city_tier'] = df_feat['city'].apply(city_tier)

In [ ]:
df_feat.head()

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title,claim,age_group,lifestyle_risk,city_tier
0,60.0,male,64.0,24.3,NoDisease,1.0,0.0,NewYork,72.0,0.0,0.0,Actor,13112.6,senior,low,1
1,49.0,female,75.0,22.6,NoDisease,1.0,0.0,Boston,78.0,1.0,1.0,Engineer,9567.0,middle_aged,low,1
2,32.0,female,64.0,17.8,Epilepsy,2.0,1.0,Phildelphia,88.0,1.0,1.0,Academician,32734.2,adult,medium,2
3,61.0,female,53.0,36.4,NoDisease,1.0,1.0,Pittsburg,72.0,1.0,0.0,Chef,48517.6,senior,high,2
4,19.0,female,50.0,20.6,NoDisease,0.0,0.0,Buffalo,82.0,1.0,0.0,HomeMakers,1731.7,young,low,2


In [ ]:
df_feat.drop(columns=['age', 'weight', 'smoker', 'city'], inplace=True)

In [ ]:
df_feat.head()

,sex,bmi,hereditary_diseases,no_of_dependents,bloodpressure,diabetes,regular_ex,job_title,claim,age_group,lifestyle_risk,city_tier
0,male,24.3,NoDisease,1.0,72.0,0.0,0.0,Actor,13112.6,senior,low,1
1,female,22.6,NoDisease,1.0,78.0,1.0,1.0,Engineer,9567.0,middle_aged,low,1
2,female,17.8,Epilepsy,2.0,88.0,1.0,1.0,Academician,32734.2,adult,medium,2
3,female,36.4,NoDisease,1.0,72.0,1.0,0.0,Chef,48517.6,senior,high,2
4,female,20.6,NoDisease,0.0,82.0,1.0,0.0,HomeMakers,1731.7,young,low,2


In [ ]:
df_feat.columns

Index(['sex', 'bmi', 'hereditary_diseases', 'no_of_dependents',
       'bloodpressure', 'diabetes', 'regular_ex', 'job_title', 'claim',
       'age_group', 'lifestyle_risk', 'city_tier'],
      dtype='object')

In [ ]:
df_feat.dtypes

,0
sex,object
bmi,float64
hereditary_diseases,object
no_of_dependents,float64
bloodpressure,float64
diabetes,float64
regular_ex,float64
job_title,object
claim,float64
age_group,object


In [ ]:
# select features and target
x = df_feat[['sex', 'bmi', 'hereditary_diseases', 'no_of_dependents'
       ,'bloodpressure', 'diabetes', 'regular_ex', 'job_title',
       'age_group', 'lifestyle_risk', 'city_tier']]
y = df_feat['claim']


In [ ]:
x

,sex,bmi,hereditary_diseases,no_of_dependents,bloodpressure,diabetes,regular_ex,job_title,age_group,lifestyle_risk,city_tier
0,male,24.3,NoDisease,1.0,72.0,0.0,0.0,Actor,senior,low,1
1,female,22.6,NoDisease,1.0,78.0,1.0,1.0,Engineer,middle_aged,low,1
2,female,17.8,Epilepsy,2.0,88.0,1.0,1.0,Academician,adult,medium,2
3,female,36.4,NoDisease,1.0,72.0,1.0,0.0,Chef,senior,high,2
4,female,20.6,NoDisease,0.0,82.0,1.0,0.0,HomeMakers,young,low,2
...,...,...,...,...,...,...,...,...,...,...,...
14995,male,28.3,NoDisease,1.0,54.0,1.0,0.0,FilmMaker,adult,medium,2
14996,male,29.6,NoDisease,4.0,64.0,1.0,0.0,Student,adult,medium,1
14997,male,33.3,NoDisease,0.0,52.0,1.0,0.0,FashionDesigner,young,medium,1
14998,male,36.7,NoDisease,0.0,70.0,1.0,0.0,Farmer,middle_aged,medium,2


In [ ]:
y

,claim
0,13112.6
1,9567.0
2,32734.2
3,48517.6
4,1731.7
...,...
14995,21082.2
14996,7512.3
14997,1391.5
14998,9144.6


In [ ]:
# define categorical and numerical features
cat_cols = ['sex', 'hereditary_diseases', 'job_title', 'age_group', 'lifestyle_risk']
num_cols = ['bmi', 'no_of_dependents', 'bloodpressure', 'diabetes', 'regular_ex', 'city_tier']

In [ ]:
# create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

In [ ]:
# best_model = RandomForestRegressor(
#     n_estimators=287,
#     max_depth=None,
#     max_features='log2',
#     min_samples_leaf=1,
#     min_samples_split=5,
#     random_state=42
# )

In [ ]:
lgb_model = LGBMRegressor(
    n_estimators=684,
    learning_rate=0.2066,
    max_depth=13,
    num_leaves=70,
    min_child_samples=12,
    colsample_bytree=0.75,
    subsample=0.87,
    random_state=42,
)

In [ ]:
# create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regression', lgb_model)
])

In [ ]:
# split data and train model
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003261 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 416
[LightGBM] [Info] Number of data points in the train set: 12000, number of used features: 60
[LightGBM] [Info] Start training from score 13442.573373
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['sex', 'hereditary_diseases',
                                                   'job_title', 'age_group',
                                                   'lifestyle_risk']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'no_of_dependents',
                                                   'bloodpressure', 'diabetes',
                                                   'regular_ex',
                                                   'city_tier'])])),
                ('regression',
                 LGBMRegressor(colsample_bytree=0.75, learning_rate=0.2066,
                               max_depth=13, min_child_samples=12,
                               n_estimators=684, num_leaves=70, random_state=42,
                               subsample=0.87))])

In [ ]:
# from sklearn.model_selection import RandomizedSearchCV
# from scipy.stats import randint, uniform

# param_dist = {
#     'regression__n_estimators': randint(200, 800),
#     'regression__num_leaves': randint(20, 80),
#     'regression__max_depth': randint(5, 20),
#     'regression__learning_rate': uniform(0.01, 0.2),
#     'regression__min_child_samples': randint(10, 50),
#     'regression__subsample': uniform(0.6, 0.4),
#     'regression__colsample_bytree': uniform(0.6, 0.4)
# }

# random_search_lgb = RandomizedSearchCV(
#     pipeline,
#     param_distributions=param_dist,
#     n_iter=30,
#     cv=3,
#     scoring='r2',
#     n_jobs=-1,
#     verbose=2,
#     random_state=42
# )

# random_search_lgb.fit(X_train, y_train)

# print("Best Parameters:", random_search_lgb.best_params_)
# print("Best CV R²:", random_search_lgb.best_score_)


In [ ]:
# predict and evaluate the model
y_pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
print(f"MAE : {mae:.2f}")
print(f"MSE : {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

MAE : 329.95
MSE : 3786339.86
RMSE: 1945.85
R²  : 0.9731


In [ ]:
import pickle

# save the trained pipeline using pickle
pickle_model_path = 'model.pkl'
with open(pickle_model_path, 'wb') as f:
  pickle.dump(pipeline, f)

In [ ]:
import sklearn
print(sklearn.__version__)

1.6.1


In [ ]:
df['bloodpressure'].describe()


,bloodpressure
count,15000.000000
mean,68.650133
std,19.418515
min,0.000000
25%,64.000000
50%,71.000000
75%,80.000000
max,122.000000
